# Anomaly Detection — Ray Parallel + MLflow Tracking

This notebook reuses the `oxy` anomaly-detection functions and parallelises them
across a Ray cluster, with every model run tracked as an MLflow child run.

**Pipeline:**
1. Feature engineering (processor + FeatureFactory) — per-sensor, Ray-parallel
2. Model scoring (run_model functions) — per-sensor × per-model, Ray-parallel
3. Consensus aggregation + cross-sensor analysis
4. All results tracked in MLflow

In [49]:
import sys, os
from pathlib import Path

# ── Detect environment ──
# K8s pods (JupyterHub + Ray): oxy mounted at /mnt/oxy via hostPath
# Local dev: use the source checkout directly
OXY_ROOT = Path(os.environ.get("OXY_ROOT", "/mnt/oxy"))
if not (OXY_ROOT / "modules" / "anomaly-detection" / "src").exists():
    OXY_ROOT = Path("/home/rami/Work/kyper/oxy")

ANOMALY_DIR = OXY_ROOT / "modules" / "anomaly-detection"
sys.path.insert(0, str(ANOMALY_DIR))
print(f"OXY_ROOT:    {OXY_ROOT}")
print(f"ANOMALY_DIR: {ANOMALY_DIR}")

OXY_ROOT:    /home/rami/Work/kyper/oxy
ANOMALY_DIR: /home/rami/Work/kyper/oxy/modules/anomaly-detection


In [50]:
import yaml
import pandas as pd
import numpy as np
import ray
import mlflow

# Reuse oxy functions directly — no modifications
from src.feature_factory import FeatureFactory
from src.visualizer import save_diagnostic
from src.cross_sensor import run_cross_sensor
from src.feature_importance import compute_and_save as save_feature_importance
from algorithms.copod import run_model as copod_run
from algorithms.ecod import run_model as ecod_run
from algorithms.lof import run_model as lof_run
from algorithms.isolation_forest import run_model as iforest_run
from algorithms.loda import run_model as loda_run
from algorithms.hbos import run_model as hbos_run

In [51]:
MODEL_REGISTRY = {
    "copod":            copod_run,
    "ecod":             ecod_run,
    "lof":              lof_run,
    "isolation_forest": iforest_run,
    "loda":             loda_run,
    "hbos":             hbos_run,
}

## 1. Configuration

In [52]:
# Load config from oxy
with open(ANOMALY_DIR / "config.yaml") as f:
    cfg = yaml.safe_load(f)

DATA_DIR = (ANOMALY_DIR / cfg["data"]["root"]).resolve()
ACTIVE_MODELS = cfg["active_models"]

# In K8s the hostPath mount is read-only — write features/results to home dir
if os.environ.get("OXY_ROOT"):
    FEATURES_DIR = Path.home() / "results" / "anomaly-detection" / "features"
    RESULTS_DIR  = Path.home() / "results" / "anomaly-detection" / "detections"
else:
    FEATURES_DIR = (ANOMALY_DIR / cfg["data"]["features"]).resolve()
    RESULTS_DIR  = (ANOMALY_DIR / cfg["data"]["results"]).resolve()

FEATURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data root:     {DATA_DIR}")
print(f"Features dir:  {FEATURES_DIR}")
print(f"Results dir:   {RESULTS_DIR}")
print(f"Active models: {[m['name'] for m in ACTIVE_MODELS]}")

Data root:     /home/rami/Work/kyper/oxy/data
Features dir:  /home/rami/Work/kyper/oxy/results/anomaly-detection/features
Results dir:   /home/rami/Work/kyper/oxy/results/anomaly-detection/detections
Active models: ['copod', 'ecod', 'lof', 'isolation_forest', 'loda', 'hbos']


## 2. Connect to Ray & MLflow

In [ ]:
import os
import atexit

# Shared pip deps needed by Ray workers for anomaly-detection tasks
RUNTIME_PIP = [
    "pandas", "numpy<2", "scikit-learn", "pyod",
    "matplotlib", "xgboost", "statsmodels", "pyyaml", "mlflow>=3.11",
]

# Disconnect any stale Ray session from a previous kernel run
if ray.is_initialized():
    ray.shutdown()

# Ray — connect to the cluster (RAY_ADDRESS set by JupyterHub pre_spawn_hook)
# Falls back to local for development outside the cluster
ray_address = os.environ.get("RAY_ADDRESS", None)
if ray_address:
    RUNTIME_ENV = {
        "pip": RUNTIME_PIP,
        "working_dir": str(ANOMALY_DIR),
        "excludes": ["__pycache__", "*.pyc"],
    }
    ray.init(address=ray_address, runtime_env=RUNTIME_ENV, ignore_reinit_error=True)
else:
    ray.init(
        ignore_reinit_error=True,
        runtime_env={"working_dir": str(ANOMALY_DIR)},
    )

atexit.register(ray.shutdown)
print(f"Ray cluster: {ray.cluster_resources()}")

# ── MLflow ───────────────────────────────────────────────────────────────
# Cluster: MLFLOW_TRACKING_URI is injected by JupyterHub
# Local:   anchored SQLite DB next to this notebook (no stale file store)
mlflow_uri = os.environ.get("MLFLOW_TRACKING_URI")
print(f"MLflow tracking: {mlflow_uri}")
if not mlflow_uri:
    _mlflow_db = (OXY_ROOT.parent / "kyper-framework" / "mlflow.db").resolve()
    mlflow_uri = f"sqlite:///{_mlflow_db}"
mlflow.set_tracking_uri(mlflow_uri)

EXPERIMENT_NAME = "anomaly_detection_ray_parallel"
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"MLflow tracking: {mlflow_uri}")
print(f"MLflow experiment: {EXPERIMENT_NAME}")


2026-04-17 09:11:10,044	INFO worker.py:2003 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
2026-04-17 09:11:10,063	INFO packaging.py:691 -- Creating a file package for local module '/home/rami/Work/kyper/oxy/modules/anomaly-detection'.
2026-04-17 09:11:10,071	INFO packaging.py:463 -- Pushing file package 'gcs://_ray_pkg_e87dac2138e5b8cb.zip' (0.56MiB) to Ray cluster...
2026-04-17 09:11:10,076	INFO packaging.py:476 -- Successfully pushed file package 'gcs://_ray_pkg_e87dac2138e5b8cb.zip'.


Ray cluster: {'memory': 27194342605.0, 'GPU': 1.0, 'node:__internal_head__': 1.0, 'accelerator_type:RTX': 1.0, 'node:172.19.119.250': 1.0, 'CPU': 32.0, 'object_store_memory': 11654718259.0}
MLflow tracking: /home/rami/mlflow-runs
MLflow experiment: anomaly_detection_ray_parallel


## 3. Feature Engineering — Ray Parallel

Discover raw sensor CSVs, generate features in parallel using `FeatureFactory`.

In [54]:
# Discover raw sensor CSVs (same filtering as processor.py)
EXCLUDE = {"hust", "NAB", "downtime"}
raw_csvs = sorted(
    p for p in DATA_DIR.rglob("*.csv")
    if not any(part in EXCLUDE for part in p.parts)
)
print(f"Found {len(raw_csvs)} raw sensor CSVs")

Found 282 raw sensor CSVs


In [55]:
@ray.remote
def build_features(raw_path_str: str, data_dir_str: str, features_dir_str: str) -> str:
    """Generate features for a single sensor CSV. Returns the feature file path."""
    raw_path = Path(raw_path_str)
    data_dir = Path(data_dir_str)
    features_dir = Path(features_dir_str)

    rel = raw_path.relative_to(data_dir)
    feature_path = features_dir / rel

    # Skip if features are already up-to-date
    if feature_path.exists() and feature_path.stat().st_mtime > raw_path.stat().st_mtime:
        return str(feature_path)

    df = pd.read_csv(raw_path, index_col=0, parse_dates=True)
    if df.shape[1] != 1:
        return ""  # skip multi-column files

    factory = FeatureFactory()
    features = factory.transform(df)
    feature_path.parent.mkdir(parents=True, exist_ok=True)
    features.to_csv(feature_path)
    return str(feature_path)

In [56]:
%%time
# Launch feature engineering in parallel
feature_futures = [
    build_features.remote(str(p), str(DATA_DIR), str(FEATURES_DIR))
    for p in raw_csvs
]
feature_paths = ray.get(feature_futures)
feature_paths = [p for p in feature_paths if p]  # drop empties
print(f"Feature files ready: {len(feature_paths)}")

Feature files ready: 281
CPU times: user 134 ms, sys: 59.2 ms, total: 194 ms
Wall time: 5.32 s


## 4. Anomaly Detection — Ray Parallel + MLflow Tracking

For each sensor, run all 6 models in parallel. Each (sensor, model) pair
is a Ray task that logs to MLflow as a nested child run.

In [57]:
@ray.remote
def run_single_model(
    feature_path_str: str,
    model_name: str,
    model_kwargs: dict,
    mlflow_uri: str,
    experiment_name: str,
    parent_run_id: str,
) -> dict:
    """
    Run one model on one sensor CSV.
    Logs params, metrics to MLflow as a nested run.
    Returns {sensor, model, n_anomalies, scores, labels}.
    """
    import mlflow as _mlflow
    _mlflow.set_tracking_uri(mlflow_uri)
    _mlflow.set_experiment(experiment_name)

    feature_path = Path(feature_path_str)
    sensor = feature_path.stem
    machine = str(feature_path.parent.name)

    X_df = pd.read_csv(feature_path, index_col=0, parse_dates=True)

    # Dispatch to the correct model
    model_fn = {
        "copod": copod_run, "ecod": ecod_run, "lof": lof_run,
        "isolation_forest": iforest_run, "loda": loda_run, "hbos": hbos_run,
    }[model_name]

    scores, labels = model_fn(X_df, **model_kwargs)
    n_anomalies = int(labels.sum())
    anomaly_rate = float(labels.mean())

    # Log to MLflow as a child run
    with _mlflow.start_run(run_name=f"{sensor}_{model_name}", nested=True, parent_run_id=parent_run_id):
        _mlflow.log_params({
            "sensor": sensor,
            "machine": machine,
            "model": model_name,
            **{k: str(v) for k, v in model_kwargs.items()},
        })
        _mlflow.log_metrics({
            "n_anomalies": n_anomalies,
            "anomaly_rate": anomaly_rate,
            "n_samples": len(X_df),
            "score_mean": float(scores.mean()),
            "score_std": float(scores.std()),
        })

    return {
        "feature_path": feature_path_str,
        "sensor": sensor,
        "model": model_name,
        "n_anomalies": n_anomalies,
        "anomaly_rate": anomaly_rate,
        "scores": scores.tolist(),
        "labels": labels.tolist(),
    }

In [58]:
feature_csvs = sorted(FEATURES_DIR.rglob("*.csv"))
print(f"Feature CSVs to process: {len(feature_csvs)}")
print(f"Models per sensor: {len(ACTIVE_MODELS)}")
print(f"Total tasks: {len(feature_csvs) * len(ACTIVE_MODELS)}")

Feature CSVs to process: 281
Models per sensor: 6
Total tasks: 1686


In [59]:
%%time
# ── Parent run: anomaly detection across all sensors × models ────────────
import time as _time

with mlflow.start_run(run_name="parallel-run-all") as parent_run:
    parent_run_id = parent_run.info.run_id
    _t0 = _time.perf_counter()

    # ── Tags: environment + provenance ───────────────────────────────────
    mlflow.set_tags({
        "notebook": "anomaly_detection_ray_parallel",
        "ray_address": os.environ.get("RAY_ADDRESS", "local"),
        "ray_version": ray.__version__,
        "oxy_root": str(OXY_ROOT),
        "config_file": str(ANOMALY_DIR / "config.yaml"),
        "env": "cluster" if os.environ.get("RAY_ADDRESS") else "local",
    })

    # ── Params: full pipeline config ─────────────────────────────────────
    mlflow.log_params({
        "n_sensors": len(feature_csvs),
        "n_models": len(ACTIVE_MODELS),
        "total_tasks": len(feature_csvs) * len(ACTIVE_MODELS),
        "model_names": ",".join(m["name"] for m in ACTIVE_MODELS),
        "features_dir": str(FEATURES_DIR),
        "results_dir": str(RESULTS_DIR),
        "data_dir": str(DATA_DIR),
    })

    # ── Log per-model hyperparams as nested params ───────────────────────
    for m_cfg in ACTIVE_MODELS:
        for k, v in m_cfg.items():
            if k != "name":
                mlflow.log_param(f"model.{m_cfg['name']}.{k}", v)

    # ── Log config.yaml as artifact ──────────────────────────────────────
    config_path = ANOMALY_DIR / "config.yaml"
    if config_path.exists():
        mlflow.log_artifact(str(config_path), artifact_path="config")

    # ── Log datasets ─────────────────────────────────────────────────────
    try:
        feature_sample = pd.read_csv(feature_csvs[0], index_col=0, parse_dates=True)
        ds = mlflow.data.from_pandas(
            feature_sample,
            source=str(FEATURES_DIR),
            name="sensor-features",
        )
        mlflow.log_input(ds, context="training")
    except Exception:
        pass  # dataset logging is best-effort

    # ── Launch all (sensor × model) tasks ────────────────────────────────
    futures = []
    for fpath in feature_csvs:
        for model_cfg in ACTIVE_MODELS:
            model_name = model_cfg["name"]
            kwargs = {k: v for k, v in model_cfg.items() if k != "name"}
            futures.append(
                run_single_model.remote(
                    str(fpath), model_name, kwargs,
                    mlflow_uri, EXPERIMENT_NAME, parent_run_id,
                )
            )

    print(f"Submitted {len(futures)} Ray tasks...")
    results = ray.get(futures)
    _elapsed = _time.perf_counter() - _t0
    print(f"Completed {len(results)} tasks in {_elapsed:.1f}s.")

    # ── Summary metrics ──────────────────────────────────────────────────
    total_anomalies = sum(r["n_anomalies"] for r in results)
    anomaly_rates = [r["anomaly_rate"] for r in results]
    mlflow.log_metrics({
        "total_anomalies": total_anomalies,
        "mean_anomaly_rate": float(np.mean(anomaly_rates)),
        "median_anomaly_rate": float(np.median(anomaly_rates)),
        "max_anomaly_rate": float(np.max(anomaly_rates)),
        "min_anomaly_rate": float(np.min(anomaly_rates)),
        "n_tasks_completed": len(results),
        "wall_time_seconds": round(_elapsed, 2),
    })

    # ── Per-model aggregate metrics ──────────────────────────────────────
    from collections import Counter
    model_counts = Counter(r["model"] for r in results)
    for model_name in model_counts:
        model_results = [r for r in results if r["model"] == model_name]
        mlflow.log_metrics({
            f"{model_name}_mean_anomaly_rate": float(np.mean([r["anomaly_rate"] for r in model_results])),
            f"{model_name}_total_anomalies": sum(r["n_anomalies"] for r in model_results),
            f"{model_name}_mean_score": float(np.mean([np.mean(r["scores"]) for r in model_results if r["scores"]])),
        })

    print(f"\nParent run: {parent_run_id}")
    print(f"Total anomalies: {total_anomalies}")
    print(f"Mean anomaly rate: {np.mean(anomaly_rates):.4f}")


/home/rami/Work/kyper/.venv/lib/python3.12/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


Submitted 1686 Ray tasks...


(run_single_model pid=93823) /home/rami/Work/kyper/.venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
(run_single_model pid=93823)   return FileStore(store_uri, store_uri)
(run_single_model pid=95840) /home/rami/Work/kyper/.venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance. [repeated 20x across

Completed 1686 tasks in 31.6s.

Parent run: f8bee030c22843edbfb166c0d802756d
Total anomalies: 39243
Mean anomaly rate: 0.0178
CPU times: user 2.96 s, sys: 994 ms, total: 3.96 s
Wall time: 31.8 s


## 5. Assemble Results & Generate Diagnostics

In [60]:
from collections import defaultdict

# Group results by feature_path
sensor_results = defaultdict(list)
for r in results:
    sensor_results[r["feature_path"]].append(r)

model_names = [m["name"] for m in ACTIVE_MODELS]

# Re-open parent run to attach result artifacts
with mlflow.start_run(run_id=parent_run.info.run_id):
    n_assembled = 0
    for feature_path_str, model_results in sensor_results.items():
        feature_path = Path(feature_path_str)
        sensor = model_results[0]["sensor"]
        X_df = pd.read_csv(feature_path, index_col=0, parse_dates=True)
        results_df = X_df.copy()

        for mr in model_results:
            results_df[f"{mr['model']}_score"] = mr["scores"]
            results_df[f"{mr['model']}_label"] = mr["labels"]

        label_cols = [f"{m}_label" for m in model_names]
        results_df["consensus"] = results_df[label_cols].sum(axis=1).astype(int)

        # Save to results dir
        rel = feature_path.relative_to(FEATURES_DIR)
        out_dir = RESULTS_DIR / rel.parent
        out_dir.mkdir(parents=True, exist_ok=True)

        csv_path = out_dir / f"{sensor}_results.csv"
        png_path = out_dir / f"{sensor}_diagnostic.png"
        results_df.to_csv(csv_path)
        save_diagnostic(results_df, model_names, sensor, png_path)

        # Log diagnostic plots as MLflow artifacts (first 10 sensors)
        if n_assembled < 10 and png_path.exists():
            mlflow.log_artifact(str(png_path), artifact_path="diagnostics")
        n_assembled += 1

    # Log a summary CSV with per-sensor consensus stats
    summary_rows = []
    for fp, mrs in sensor_results.items():
        sensor = mrs[0]["sensor"]
        total = sum(r["n_anomalies"] for r in mrs)
        rate = np.mean([r["anomaly_rate"] for r in mrs])
        summary_rows.append({"sensor": sensor, "total_anomalies": total, "mean_anomaly_rate": rate})
    summary_df = pd.DataFrame(summary_rows).sort_values("total_anomalies", ascending=False)
    summary_path = RESULTS_DIR / "sensor_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    mlflow.log_artifact(str(summary_path), artifact_path="results")

    mlflow.log_metric("n_sensors_assembled", n_assembled)

print(f"Assembled results for {n_assembled} sensors.")
print(f"Top anomaly sensors:\n{summary_df.head(10).to_string(index=False)}")


Assembled results for 281 sensors.
Top anomaly sensors:
             sensor  total_anomalies  mean_anomaly_rate
  Accelerometer1RMS              987           0.017582
  Accelerometer2RMS              987           0.017582
            Current              987           0.017582
           Pressure              987           0.017582
        Temperature              987           0.017582
       Thermocouple              987           0.017582
            Voltage              987           0.017582
Volume_Flow_RateRMS              987           0.017582
    raw_sensor_data              825           0.015785
            Current              137           0.017866


## 6. Cross-Sensor Analysis

Run cross-sensor consensus for each machine directory.

In [ ]:
# Get unique machine directories from results
machine_dirs = set()
for fpath_str in sensor_results:
    fpath = Path(fpath_str)
    rel = fpath.relative_to(FEATURES_DIR)
    machine_dir = RESULTS_DIR / rel.parent
    machine_dirs.add(machine_dir)

for mdir in sorted(machine_dirs):
    result_csvs = list(mdir.glob("*_results.csv"))
    if result_csvs:
        print(f"  Cross-sensor: {mdir.relative_to(RESULTS_DIR)} ({len(result_csvs)} sensors)")
        run_cross_sensor(mdir)

## 7. Summary

In [ ]:
# Build summary DataFrame
summary_rows = []
for r in results:
    summary_rows.append({
        "sensor": r["sensor"],
        "model": r["model"],
        "n_anomalies": r["n_anomalies"],
        "anomaly_rate": round(r["anomaly_rate"], 4),
    })

summary_df = pd.DataFrame(summary_rows)
print(f"\nTotal tasks: {len(summary_df)}")
print(f"Total anomalies: {summary_df['n_anomalies'].sum()}")
print(f"\nPer-model anomaly counts:")
print(summary_df.groupby("model")["n_anomalies"].sum().to_string())
print(f"\nMLflow parent run: {parent_run_id}")
print(f"MLflow UI: {mlflow_uri}/#/experiments")

In [ ]:
# Top sensors by anomaly rate (across all models)
pivot = summary_df.pivot_table(index="sensor", columns="model", values="anomaly_rate")
pivot["mean_rate"] = pivot.mean(axis=1)
pivot.sort_values("mean_rate", ascending=False).head(15)

In [ ]:
ray.shutdown()